In [1]:
from obspy.core import read, Stream, Trace
from obspy.signal.filter import lowpass,highpass
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from geopy.distance import great_circle, geodesic
import warnings
import glob
import re
from pyproj import CRS, Transformer
import pickle
import xml.etree.ElementTree as ET
import datetime
import obspy
from shapely.geometry import Point, Polygon
import cartopy.crs as ccrs         # to plot maps with different projections
import cartopy.feature as cfeature # to plot coastlines, land, borders, etc.
import json
with open('/Users/zoekrauss/offshore_eew_modeling/location/PB2002_boundaries.json', 'r') as f:
    plate_boundaries = json.load(f)
import matplotlib
matplotlib.rcParams['font.family'] = ['Arial']
plt.rcParams.update({'font.size': 18}) 

In [2]:
# Functions to read logs
def get_block_info(block):
    
    """
    Designed to parse one block of a dmevent xml log
    """
    
    # CORE INFO #
    mag = float(block[0][0].text)
    ev_lat = float(block[0][2].text)
    ev_lon = float(block[0][4].text)

    message_time = obspy.UTCDateTime(block.items()[7][1])

    # STATION INFO #
    sta_lats = []
    sta_lons = []
    sta_vals = []
    sta_info = []
    for sta in block[-1][0][0]:
        try: 
            sta_alg = sta.items()[1][1]
        except:
            sta_alg = 'unassoc'
        sta_lats.append(float(sta[2].text))
        sta_lons.append(float(sta[3].text))
        sta_vals.append(float(sta[1].text)) # acceleration in cm/s/s
        sta_chan = sta[0].text.split('.')[2]
        # Get distance between this station and the hypocenter
        sta_dist = geodesic([float(sta[2].text),float(sta[3].text)],[ev_lat,ev_lon]).km
        sta_info.append([float(sta[3].text),float(sta[2].text),float(sta[1].text),sta_dist,sta_alg,sta_chan])

    try:
        for sta in block[-1][0][1]:
            try: 
                sta_alg = sta.items()[1][1]
            except:
                sta_alg = 'unassoc'
            sta_lats.append(float(sta[2].text))
            sta_lons.append(float(sta[3].text))
            sta_vals.append(float(sta[1].text)) # acceleration in cm/s/s
            sta_chan = sta[0].text.split('.')[2]
            # Get distance between this station and the hypocenter
            sta_dist = geodesic([float(sta[2].text),float(sta[3].text)],[ev_lat,ev_lon]).km
            sta_info.append([float(sta[3].text),float(sta[2].text),float(sta[1].text),sta_dist,sta_alg,sta_chan])
    except:
        # print('Only one type of ground motion')
        a=1

    # FAULT INFO, if it exists! #
    vertices = []
    if len(block) > 3:
        for v in block[2][0][0][0]:
            vertices.append([float(v[1].text),float(v[0].text)])
    
        
    block_dict = {'timestamp':message_time,'mag':mag,'ev_lat':ev_lat,'ev_lon':ev_lon,'sta_info':sta_info,'vertices':vertices}

    return block_dict

def read_source_file(filename):
    """
    Read hypocenter lon, lat, and magnitude from Audrey's source text files
    """

    with open(filename,'r') as f:
        lines = f.readlines()

    magnitude = None
    for line in lines:
        if "Hypocenter X:" in line:
            match = re.search(r"X:\s*([\d.]+),\s*Y:\s*([\d.]+)", line)
            if match:
                x = float(match.group(1))
                y = float(match.group(2))
        elif line.startswith("Magnitude:") and magnitude is None:
            match = re.search(r"Magnitude:\s*([\d.]+)", line)
            if match:
                magnitude = float(match.group(1))

    # Hypocenter is given in UTM coordinates. Convert to lat/lon
    utm_crs = CRS.from_epsg(32610)  # EPSG:32610 = WGS84 / UTM zone 10N
    wgs84_crs = CRS.from_epsg(4326)  # WGS84 lat/lon
    # Create transformer
    transformer = Transformer.from_crs(utm_crs, wgs84_crs, always_xy=True)
    lon, lat = transformer.transform(x,y)
    
    return lon,lat,magnitude

def extract_xml(line):
    start = line.find('<')
    end = line.rfind('>')

    if start != -1 and end != -1 and end > start:
        xml_string = line[start:end+1]

    return xml_string

def get_xml_blocks(filename):
    """
    Pull out the relevant information from XML log files as separate blocks per each event update
    """
    xml_lines = []
    inside_xml = False
    complete_blocks = []
    
    with open(filename, "r") as file:
        for line in file:
            # if '<event_message alg_vers="3.3.7-2023-11-01" category="live"' in line:
            if '<event_message alg_vers=' in line:
                inside_xml = True
                # add_line = re.findall(r'<.*?>', line.strip()).group()
                add_line = extract_xml(line.strip())
                xml_lines = [add_line]
            elif "</event_message>" in line:
                # add_line = re.search(r'<.*?>', line.strip()).group()
                add_line = extract_xml(line.strip())
                xml_lines.append(add_line)
                inside_xml = False
                xml_string = "".join(xml_lines)
                try:
                    element = ET.fromstring(xml_string)
                    complete_blocks.append(element)
                    # print(f"Parsed block with tag: {element.tag}")
                except ET.ParseError as e:
                    print(f"Error parsing block: {e}")
            elif inside_xml:
                try:
                    add_line = extract_xml(line.strip())
                    # add_line = re.search(r'<.*?>', line.strip()).group()
                    xml_lines.append(add_line)
                except:
                    continue

    return complete_blocks

def get_block_info_dm(block):
    
    """
    Designed to parse one block of a dmevent xml log
    """
    
    # CORE INFO #
    mag = float(block[0][0].text)
    ev_lat = float(block[0][2].text)
    ev_lon = float(block[0][4].text)

    message_time = obspy.UTCDateTime(block.items()[7][1])

    # STATION INFO #
    sta_lats = []
    sta_lons = []
    sta_vals = []
    sta_info = []
    for sta in block[-1][0][0]:
        try: 
            sta_alg = sta.items()[1][1]
        except:
            sta_alg = 'unassoc'
        sta_lats.append(float(sta[2].text))
        sta_lons.append(float(sta[3].text))
        sta_vals.append(float(sta[1].text)) # acceleration in cm/s/s
        sta_chan = sta[0].text.split('.')[2]
        # Get distance between this station and the hypocenter
        sta_dist = geodesic([float(sta[2].text),float(sta[3].text)],[ev_lat,ev_lon]).km
        sta_info.append([float(sta[3].text),float(sta[2].text),float(sta[1].text),sta_dist,sta_alg,sta_chan])

    # FAULT INFO, if it exists! #
    vertices = []
    if len(block) > 3:
        for v in block[2][0][0][0]:
            vertices.append([float(v[1].text),float(v[0].text)])
    
        
    block_dict = {'timestamp':message_time,'mag':mag,'ev_lat':ev_lat,'ev_lon':ev_lon,'sta_info':sta_info,'vertices':vertices}

    return block_dict

def get_replay_start(filename):
    """
    Read tankplayer log and extract start time
    """
    
    # Read only the first line from the file
    with open(filename, "r") as f:
        for _ in range(10):  # scan up to first 10 lines
            line = f.readline()
            match = re.search(r'UTC_(\d{8})_(\d{2}:\d{2}:\d{2})', line)
            if match:
                date_str = match.group(1)
                time_str = match.group(2)
                utc_time = datetime.datetime.strptime(f"{date_str} {time_str}", "%Y%m%d %H:%M:%S")
                break

    return obspy.UTCDateTime(utc_time)

def get_block_info_gm(block):
    """
    Read a eqinfo2gm log
    Only parse each the block if it is a contour message
    Save relevant info as a dictionary

    If contour message, returns the dictionary
    If map message, returns none
    """

    
    finder = False
    contour  = False
    mapp = False
    
    # CORE INFO #
    mag = float(block[0][0].text)
    ev_lat = float(block[0][2].text)
    ev_lon = float(block[0][4].text)
    
    message_time = obspy.UTCDateTime(block.items()[7][1])
    
    # Did FinDer contribute to this event? If so, there will be finite fault info
    if len(block[1]) > 2:
        finder = True
    
    # Is this a map or contour block?
    if 'contour' in block.items()[2][1]:
        contour = True
    elif 'map' in block.items()[2][1]:
        mapp = True
        return []

    # FAULT INFO, if it exists! #
    fault = []
    if finder:
        try:
            for v in block[2][0][0][0]:
                fault.append([float(v[1].text),float(v[0].text)])
        except:
            # print('no fault info')
            a=1

    # If contour, extract each output contour AND station info
    polygons = []
    mmi = []
    if contour:
        # Each contour has a polygon defined by vertices, and a corresponding value
        for con in block[-1][1]:
            mmi.append(float(con[0].text))
            vertices = con[3].text.split(' ')
            vertices = [v.split(',') for v in vertices]
            vertices = [[float(v[0]),float(v[1])] for v in vertices]
            polygons.append(vertices)

        # Also save the station info!
        sta_lats = []
        sta_lons = []
        sta_vals = []
        sta_info = []
        for sta in block[-1][0][0]:
            try: 
                sta_alg = sta.items()[1][1]
            except:
                sta_alg = 'unassoc'
            sta_lats.append(float(sta[2].text))
            sta_lons.append(float(sta[3].text))
            sta_vals.append(float(sta[1].text)) # acceleration in cm/s/s
            sta_chan = sta[0].text.split('.')[2]
            # Get distance between this station and the hypocenter
            sta_dist = geodesic([float(sta[2].text),float(sta[3].text)],[ev_lat,ev_lon]).km
            sta_info.append([float(sta[3].text),float(sta[2].text),float(sta[1].text),sta_dist,sta_alg,sta_chan])

        block_dict = {'timestamp':message_time,'mag':mag,'ev_lat':ev_lat,'ev_lon':ev_lon,'contour_mmi':mmi,'contour_vertices':polygons,'fault_vertices':fault,'sta_info':sta_info,}

        return block_dict

def get_replay_start(filename):
    """
    Read tankplayer log and extract start time
    """
    
    # Read only the first line from the file
    with open(filename, "r") as f:
        for _ in range(10):  # scan up to first 10 lines
            line = f.readline()
            match = re.search(r'UTC_(\d{8})_(\d{2}:\d{2}:\d{2})', line)
            if match:
                date_str = match.group(1)
                time_str = match.group(2)
                utc_time = datetime.datetime.strptime(f"{date_str} {time_str}", "%Y%m%d %H:%M:%S")
                break

    return obspy.UTCDateTime(utc_time)

In [3]:
# First EPIC solution - time and mag
# First finder solution - time and mag
# Largest mag
# Time to largest mag

In [22]:
base_path = '/Users/zoekrauss/shakealert_8hz/replays/mendocino/mendocino_first10_downsampled/mendocino/runs/*'
base_path = '/Users/zoekrauss/shakealert_8hz/replays/mendocino/mendocino_first10_excluded/mendocino/runs/*'
# base_path = '/Users/zoekrauss/shakealert_8hz/replays/mendocino/mendocino_fullrate/runs/*'

paths = glob.glob(base_path)

In [23]:
paths
# paths = paths[1:]

['/Users/zoekrauss/shakealert_8hz/replays/mendocino/mendocino_first10_excluded/mendocino/runs/20260914_2',
 '/Users/zoekrauss/shakealert_8hz/replays/mendocino/mendocino_first10_excluded/mendocino/runs/20260914_3',
 '/Users/zoekrauss/shakealert_8hz/replays/mendocino/mendocino_first10_excluded/mendocino/runs/20260911_2',
 '/Users/zoekrauss/shakealert_8hz/replays/mendocino/mendocino_first10_excluded/mendocino/runs/20260914_1']

In [24]:
first_epic_times=[]
first_epic_mags=[]
max_epic_mags=[]
first_finder_times=[]
first_finder_mags=[]
max_finder_mags=[]
max_finder_times = []

for j in range(4):

    print(j)
    filename = glob.glob(paths[j] + '/**/tankplayer*')[0]
    replay_start = get_replay_start(filename)   
    
    # EPIC
    filename = glob.glob(paths[j]+'/*/epic_2*')[0]
    complete_blocks = get_xml_blocks(filename)
    mag_times=[];mag_mags=[]
    for i,bl in enumerate(complete_blocks):
        block = get_block_info(bl)
        mag_time=block['timestamp']-replay_start
        mag_mag=block['mag']
        mag_times.append(mag_time);mag_mags.append(mag_mag)
    try:
        first_epic_times.append(np.min(mag_times))
        first_epic_mags.append(mag_mags[0])
        max_epic_mags.append(np.max(mag_mags))
    except:
        first_epic_times.append(np.nan)
        first_epic_mags.append(np.nan)
        max_epic_mags.append(np.nan)
    
    # FinDer
    filename = glob.glob(paths[j]+'/*/finder*')[0]
    print(filename)
    complete_blocks = get_xml_blocks(filename)
    mag_times=[];mag_mags=[]
    for i,bl in enumerate(complete_blocks):
        block = get_block_info(bl)
        mag_time=block['timestamp']-replay_start
        mag_mag=block['mag']
        mag_times.append(mag_time);mag_mags.append(mag_mag)
    first_finder_times.append(np.min(mag_times))
    first_finder_mags.append(mag_mags[0])
    max_finder_mags.append(np.max(mag_mags))
    # Time to within 0.2 units of max mag
    target_mag = np.max(mag_mags) - 0.2
    target_ind = [k for k,m in enumerate(mag_mags) if m>=target_mag][0]
    max_finder_times.append(mag_times[target_ind])

0
/Users/zoekrauss/shakealert_8hz/replays/mendocino/mendocino_first10_excluded/mendocino/runs/20260914_2/logs/finder_20260914.log
1
/Users/zoekrauss/shakealert_8hz/replays/mendocino/mendocino_first10_excluded/mendocino/runs/20260914_3/logs/finder_20260914.log
2
/Users/zoekrauss/shakealert_8hz/replays/mendocino/mendocino_first10_excluded/mendocino/runs/20260911_2/logs/finder_20260911.log
3
/Users/zoekrauss/shakealert_8hz/replays/mendocino/mendocino_first10_excluded/mendocino/runs/20260914_1/logs/finder_20260914.log


In [25]:
print('First epic detection at '+str(np.nanmean(first_epic_times))+' s')
print('First epic mag '+str(np.nanmean(first_epic_mags)))
print('Largest epic mag '+str(np.nanmean(max_epic_mags)))
print('First finder detection at '+str(np.nanmean(first_finder_times))+' s')
print('First finder mag '+str(np.nanmean(first_finder_mags)))
print('Largest finder mag '+str(np.nanmean(max_finder_mags)))
print('Finder mag got within 0.2 units of max mag at '+str(np.nanmean(max_finder_times)) + str(' s'))

First epic detection at 134.62025 s
First epic mag 5.1801
Largest epic mag 7.392175
First finder detection at 141.4505 s
First finder mag 5.5
Largest finder mag 6.3
Finder mag got within 0.2 units of max mag at 150.4175 s


In [26]:
print('First epic detection at '+str(np.nanmean(first_epic_times))+' s')
print('First epic mag '+str(np.nanmean(first_epic_mags)))
print('Largest epic mag '+str(np.nanmean(max_epic_mags)))
print('First finder detection at '+str(np.nanmean(first_finder_times))+' s')
print('First finder mag '+str(np.nanmean(first_finder_mags)))
print('Largest finder mag '+str(np.nanmean(max_finder_mags)))
print('Finder mag got within 0.2 units of max mag at '+str(np.nanmean(max_finder_times)) + str(' s'))

First epic detection at 134.62025 s
First epic mag 5.1801
Largest epic mag 7.392175
First finder detection at 141.4505 s
First finder mag 5.5
Largest finder mag 6.3
Finder mag got within 0.2 units of max mag at 150.4175 s


In [21]:
print('First epic detection at '+str(np.nanmean(first_epic_times))+' s')
print('First epic mag '+str(np.nanmean(first_epic_mags)))
print('Largest epic mag '+str(np.nanmean(max_epic_mags)))
print('First finder detection at '+str(np.nanmean(first_finder_times))+' s')
print('First finder mag '+str(np.nanmean(first_finder_mags)))
print('Largest finder mag '+str(np.nanmean(max_finder_mags)))
print('Finder mag got within 0.2 units of max mag at '+str(np.nanmean(max_finder_times)) + str(' s'))

First epic detection at 134.36374999999998 s
First epic mag 5.221525000000001
Largest epic mag 7.412175
First finder detection at 140.7305 s
First finder mag 5.4
Largest finder mag 6.300000000000001
Finder mag got within 0.2 units of max mag at 150.14325 s
